### Импорты

In [ ]:
import pandas as pd
import numpy as np
import datetime
import networkx 
from openai import OpenAI
from tqdm import tqdm
import requests
import json
import sys
import re
from pprint import pprint
from keybert import KeyBERT

from openai import OpenAI

from sentence_transformers import SentenceTransformer
import hdbscan
from collections import Counter
from sklearn.cluster import DBSCAN

from sentence_transformers import SentenceTransformer
import networkx as nx

from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
kw_model = KeyBERT(model="paraphrase-multilingual-MiniLM-L12-v2")

### Парсинг сырых вакансий

In [ ]:
url = "https://api.hh.ru/vacancies"

params = {
    "text": "аналитик данных",
    "professional_role" : [164], #164 pa 165 ds
    "area": 1,  # Москва
    "per_page": 100,
    "only_with_salary": False,
    "page" : 0,
    "order_by": "publication_time"
}

In [ ]:
vacancies_raw_lst = []
page = 0

for i in range(1, 4):
    params['page'] = page
    response = requests.get(url, params = params).json()
    vacancies_raw_lst.extend(response['items'] )
    page += 1

### Списки требований / обязанностей / key skills

In [ ]:
vacancies_req_lst = [{'id':i.get('id'), 'req':i.get('snippet', {}).get('requirement', None) } for i in vacancies_raw_lst]
vacancies_req_lst[0]

In [ ]:
vacancies_res_lst = [{'id':i.get('id'), 'res':i.get('snippet', {}).get('responsibility', None) } for i in vacancies_raw_lst]
vacancies_res_lst[0]

In [ ]:
vacancies_skills_lst = []
for id in tqdm([i['id'] for i in vacancies_raw_lst]):
    try:
        skills_lst_pre = requests.get(f"https://api.hh.ru/vacancies/{id}").json()['key_skills']
        skills_dict = {'id' : id, 'skills' : [i.get('name') for i in skills_lst_pre] }
        vacancies_skills_lst.append(skills_dict)
    except:
        vacancies_skills_lst.append({'id' : id, 'skills' : [] })

In [ ]:
df_skills = pd.DataFrame(vacancies_skills_lst).explode('skills')

In [ ]:
df_skills.groupby(['skills'], as_index = False)['id'].count().sort_values(by = 'id', ascending = False).head(30)

### Ключевые требования / обязанности / key skills

##### Требования

In [ ]:
def clean_requirement(text):
    text = re.sub(r"<.*?>", " ", text)  # убрать HTML
    text = text.lower()
    return text

def split_phrases(text):
    return re.split(r",|;|•|\n", text)

In [ ]:
req_keywords = []

for i in tqdm(vacancies_req_lst):
    if i.get('req', 0) in [None, 0]:
        continue
    else:
        text = clean_requirement(i.get('req') )
        phrases = split_phrases(text)
        for phrase in phrases:
            keywords = kw_model.extract_keywords(
                phrase, 
                keyphrase_ngram_range = (2, 4),  # uni-, bi-, триграммы
                stop_words=None,
                top_n = 3)
            # keywords = список кортежей (фраза, score)
            req_keywords.extend([kw for kw, score in keywords])

In [ ]:
len(req_keywords)

##### Обязанности

In [ ]:
res_keywords = []

for i in tqdm(vacancies_res_lst):
    if i.get('res', 0) in [None, 0]:
        continue
    else:
        text = clean_requirement(i.get('res') )
        phrases = split_phrases(text)
        for phrase in phrases:
            keywords = kw_model.extract_keywords(
                phrase, 
                keyphrase_ngram_range = (2, 4),  # uni-, bi-, триграммы
                stop_words=None,
                top_n = 3)
            # keywords = список кортежей (фраза, score)
            res_keywords.extend([kw for kw, score in keywords])

In [ ]:
len(res_keywords)

### Требования кластеризация Perplexity

In [ ]:
base_prompt_total = f"""
    Ты — эксперт по анализу вакансий и нормализации навыков для ATS-резюме.
    
    Твоя задача:
    1. Получить список коротких фраз (навыков / требований из вакансий).
    2. Объединить их в логические группы (макро-навыки).
    3. Для каждой группы придумать короткое, понятное, “человеческое” название (1–3 слова).
    4. Сгруппировать исходные фразы без изменений.
    
    ВАЖНО:
    - Не придумывай новые навыки, которых нет в списке
    - Не изменяй исходные фразы
    - Используй названия групп, которые обычно встречаются в резюме и вакансиях
    - Одна фраза должна попасть только в одну группу
    - Названия групп должны быть понятны для резюме (ATS-friendly)
    - Игнорируй дубли (оставь один раз)

    Сделай не более 20 групп
    Каждая группа должна содержать минимум 2 фразы.
    Если фраза не подходит ни к одной группе — создай группу "прочее".
    
    Формат ответа строго JSON:
    
    {{
      "название группы 1": ["фраза 1", "фраза 2"],
      "название группы 2": ["фраза 3", "фраза 4"]
    }}
    
    Вот список фраз:
"""

In [ ]:
base_prompt_common_phrases = """
Ты — эксперт по анализу текстов вакансий. 
Тебе дан список ключевых фраз (от 1 до 4 слов) из ранее спаршенных вакансий.

Твоя задача:
1. Проанализировать список и объединить фразы по смыслу.
2. Выделить топ-30 наиболее значимых макро-фраз.
3. Каждая макро-фраза должна быть максимально “человечной”, как если бы она была прямо в тексте вакансии.
4. Сохраняй **оригинальные формулировки, стиль и семантику** исходных фраз.
5. Каждая макро-фраза должна состоять максимум из 1, 2, 3, 4 слов.
6. Для каждой макро-фразы укажи, сколько исходных фраз легло в её основу.

ТРЕБОВАНИЯ:
- Используй только данные из предоставленного списка, не придумывай новые слова или технологии.
- Убирай дубли.
- Формулируй фразы так, чтобы они выглядели естественно и были понятны человеку, читающему описание вакансии.
- Если возможно, объединяй похожие фразы, но не меняй их суть.

ФОРМАТ ВЫВОДА СТРОГО JSON:

{
  "1": [ {"фраза": "<фраза 1 слово>", "count": <кол-во исходных фраз>}, ... ],
  "2": [ {"фраза": "<фраза 2 слова>", "count": <кол-во исходных фраз>}, ... ],
  "3": [ {"фраза": "<фраза 3 слова>", "count": <кол-во исходных фраз>} ... ]
  "4": [ {"фраза": "<фраза 4 слова>", "count": <кол-во исходных фраз>} ... ]
}

УСЛОВИЯ:
- Раздели макро-фразы по длине слов: 1, 2, 3.
- В каждом списке максимум 30 макро-фраз по частоте.
- НЕ используй markdown (никаких ```json).
- Ответ должен быть валидным JSON, начинающимся с { и заканчивающимся }.
- Если для какой-то категории нет фраз — верни пустой список [].

ФОРМАТ ВЫВОДА СТРОГО JSON

Список ключевых фраз:
"""

In [ ]:
prompt = base_prompt_common_phrases + "\n" + ", ".join(req_keywords)

In [ ]:
client = OpenAI(
    api_key = ,
    base_url = "https://api.perplexity.ai"
)

response = client.chat.completions.create(
    model = "sonar",
    messages = [
        {
            "role": "system",
            "content": "Ты — эксперт по анализу вакансий и нормализации навыков."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]
)

In [ ]:
with open('req_da_phrases.json', 'w') as f:
    json.dump(json.loads(response.choices[0].message.content), f, ensure_ascii=False, indent = 2)

In [ ]:
json.loads(response.choices[0].message.content)